# Deep Learning Model Selection

Three fully connected MLPs are implemented in low-level TensorFlow and compared:  
**Shallow** (1 hidden layer) · **Medium** (2 hidden layers) · **Deep** (3 hidden layers + Dropout)  

CNNs are not considered here — the input is tabular (5 structured features), not image-based.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score
warnings.filterwarnings("ignore")
tf.random.set_seed(42)

df = pd.read_csv("../data/ai_resume_screening.csv").drop(columns=["github_activity"])

X = df.drop("shortlisted", axis=1)
y = df["shortlisted"].map({"Yes": 1, "No": 0})

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

education_order = {"High School": 0, "Bachelors": 1, "Masters": 2, "PhD": 3}
X_train_enc = X_train.copy(); X_val_enc = X_val.copy(); X_test_enc = X_test.copy()
for s in [X_train_enc, X_val_enc, X_test_enc]:
    s["education_level"] = s["education_level"].map(education_order)

classes = np.array([0, 1])
cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, cw))

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_enc)
X_val_s   = scaler.transform(X_val_enc)
X_test_s  = scaler.transform(X_test_enc)

# Convert to TF tensors
X_tr = tf.constant(X_train_s, dtype=tf.float32)
y_tr = tf.constant(y_train.values.reshape(-1, 1), dtype=tf.float32)
X_vl = tf.constant(X_val_s,   dtype=tf.float32)
y_vl = tf.constant(y_val.values.reshape(-1, 1),   dtype=tf.float32)
X_ts = tf.constant(X_test_s,  dtype=tf.float32)
y_ts = tf.constant(y_test.values.reshape(-1, 1),  dtype=tf.float32)

# Sample weights for class imbalance
sample_weights = tf.constant(
    y_train.map(class_weight_dict).values.reshape(-1, 1), dtype=tf.float32
)

n_features = X_train_s.shape[1]
print(f"Setup complete. n_features = {n_features}")
print(f"Train: {X_tr.shape[0]} | Val: {X_vl.shape[0]} | Test: {X_ts.shape[0]}")

## Model Architectures

In [ ]:
def W(shape): return tf.Variable(tf.random.normal(shape, stddev=0.1))
def b(shape): return tf.Variable(tf.zeros(shape))

# Shallow: input -> 8 -> 1
W1_s = W((n_features, 8));  b1_s = b((8,))
W2_s = W((8, 1));           b2_s = b((1,))
vars_shallow = [W1_s, b1_s, W2_s, b2_s]

def forward_shallow(x, training=False):
    return tf.matmul(tf.nn.relu(tf.matmul(x, W1_s) + b1_s), W2_s) + b2_s

# Medium: input -> 16 -> 8 -> 1
W1_m = W((n_features, 16)); b1_m = b((16,))
W2_m = W((16, 8));          b2_m = b((8,))
W3_m = W((8, 1));           b3_m = b((1,))
vars_medium = [W1_m, b1_m, W2_m, b2_m, W3_m, b3_m]

def forward_medium(x, training=False):
    h1 = tf.nn.relu(tf.matmul(x,  W1_m) + b1_m)
    h2 = tf.nn.relu(tf.matmul(h1, W2_m) + b2_m)
    return tf.matmul(h2, W3_m) + b3_m

# Deep: input -> 32 -> 16 -> 8 -> 1  (Dropout 0.2 on first hidden layer)
W1_d = W((n_features, 32)); b1_d = b((32,))
W2_d = W((32, 16));         b2_d = b((16,))
W3_d = W((16, 8));          b3_d = b((8,))
W4_d = W((8, 1));           b4_d = b((1,))
vars_deep = [W1_d, b1_d, W2_d, b2_d, W3_d, b3_d, W4_d, b4_d]

def forward_deep(x, training=False):
    h1 = tf.nn.relu(tf.matmul(x,  W1_d) + b1_d)
    if training:
        h1 = tf.nn.dropout(h1, rate=0.2)
    h2 = tf.nn.relu(tf.matmul(h1, W2_d) + b2_d)
    h3 = tf.nn.relu(tf.matmul(h2, W3_d) + b3_d)
    return tf.matmul(h3, W4_d) + b4_d

print("Architectures defined.")
print(f"  Shallow: {n_features} -> 8 -> 1")
print(f"  Medium:  {n_features} -> 16 -> 8 -> 1")
print(f"  Deep:    {n_features} -> 32 -> 16 -> 8 -> 1  (Dropout 0.2)")

## Training

Mini-batch gradient descent with Adam optimizer.  
Class imbalance is handled via per-sample weights in the loss.

In [ ]:
def weighted_bce(logits, labels, weights):
    loss = tf.nn.sigmoid_cross_entropy_with_logits(labels=labels, logits=logits)
    return tf.reduce_mean(loss * weights)

def train(forward_fn, variables, epochs=60, batch_size=256, lr=0.001):
    opt = tf.optimizers.Adam(learning_rate=lr)
    n   = X_tr.shape[0]
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(epochs):
        idx = tf.random.shuffle(tf.range(n))
        Xs = tf.gather(X_tr, idx)
        ys = tf.gather(y_tr, idx)
        ws = tf.gather(sample_weights, idx)

        batch_losses = []
        for i in range(0, n, batch_size):
            Xb, yb, wb = Xs[i:i+batch_size], ys[i:i+batch_size], ws[i:i+batch_size]
            with tf.GradientTape() as tape:
                loss = weighted_bce(forward_fn(Xb, training=True), yb, wb)
            opt.apply_gradients(zip(tape.gradient(loss, variables), variables))
            batch_losses.append(loss.numpy())

        val_loss = weighted_bce(
            forward_fn(X_vl, training=False), y_vl, tf.ones_like(y_vl)
        ).numpy()
        history["train_loss"].append(float(np.mean(batch_losses)))
        history["val_loss"].append(float(val_loss))

    return history

print("Training Shallow...")
hist_shallow = train(forward_shallow, vars_shallow)
print("Training Medium...")
hist_medium  = train(forward_medium,  vars_medium)
print("Training Deep...")
hist_deep    = train(forward_deep,    vars_deep)
print("Done.")

## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
configs = [
    ("Shallow",  hist_shallow, "steelblue"),
    ("Medium",   hist_medium,  "seagreen"),
    ("Deep",     hist_deep,    "tomato"),
]
for ax, (name, hist, color) in zip(axes, configs):
    ax.plot(hist["train_loss"], label="Train",      color=color)
    ax.plot(hist["val_loss"],   label="Validation", color=color, linestyle="--", alpha=0.7)
    ax.set_title(name)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(alpha=0.3)
plt.suptitle("Training vs Validation Loss", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Evaluation — Validation Set

In [ ]:
def predict(forward_fn, X_tf, threshold=0.5):
    logits = forward_fn(X_tf, training=False)
    return (tf.sigmoid(logits).numpy().flatten() >= threshold).astype(int)

for name, fwd in [("Shallow", forward_shallow), ("Medium", forward_medium), ("Deep", forward_deep)]:
    preds = predict(fwd, X_vl)
    print(f"{name} — Validation Set")
    print(classification_report(y_val, preds, target_names=["Not Shortlisted", "Shortlisted"]))

## Comparison

In [ ]:
from sklearn.metrics import precision_score, recall_score

rows = []
for name, fwd in [("Shallow", forward_shallow), ("Medium", forward_medium), ("Deep", forward_deep)]:
    preds = predict(fwd, X_vl)
    rows.append({
        "Model":     name,
        "Precision": round(precision_score(y_val, preds, zero_division=0), 4),
        "Recall":    round(recall_score(y_val, preds), 4),
        "F1-Score":  round(f1_score(y_val, preds), 4),
        "Accuracy":  round((preds == y_val.values).mean(), 4)
    })

pd.DataFrame(rows).set_index("Model")

## Final Test Set Evaluation

In [ ]:
print("Medium Network — Test Set")
preds_test = predict(forward_medium, X_ts)
print(classification_report(y_test, preds_test, target_names=["Not Shortlisted", "Shortlisted"]))